# Module 3.3: Staged Promotion

New memories should NOT immediately influence agent behaviour. This notebook implements
a **state machine** where memories must earn trust through repeated confirmation before
they become decision-affecting.

## The State Machine

```
              ┌──────────┐     1+ confirmations      ┌──────────────┐     2+ confirmations     ┌──────────┐
  identify →  │ CANDIDATE │ ────────────────────────→  │ PROVISIONAL  │ ─────────────────────→   │ TRUSTED  │
              └──────────┘                            └──────────────┘                          └──────────┘
                   │                                        │                                        │
                   │  confidence < 0.2                      │  contradicted                          │  staleness > 180d
                   ↓                                        ↓                                        ↓
              ┌──────────────┐                        ┌──────────────┐                        ┌──────────────┐
              │  DEPRECATED  │                        │  DEPRECATED  │                        │ PROVISIONAL  │
              └──────────────┘                        └──────────────┘                        └──────────────┘
```

## Why Staged Promotion?

This is the primary defense against **memory poisoning** — where adversarial or
incorrect information gets stored and immediately influences decisions.

> *"From Untrusted Input to Trusted Memory"* (arXiv:2606.04329) — shows that agents
> writing memory aggressively are more exploitable. Staged promotion directly addresses
> this by preventing single writes from having immediate impact.

In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
import sys, os, json, asyncio
import nest_asyncio
from datetime import datetime, timezone, timedelta

sys.path.insert(0, "..")
nest_asyncio.apply()

from lifecycle_utils import (
    MemoryState, MemoryItem, PromotionConfig, PromotionEngine
)
from shared.travel_agent import create_client, SYSTEM_PROMPT

client, credential = create_client("../.env")
print("Setup complete")

## Configure the Promotion Engine

The `PromotionConfig` controls how much evidence is needed before memories
can influence agent decisions.

In [ ]:
config = PromotionConfig(
    confirmation_threshold=2,          # 2 confirms: candidate → trusted
    provisional_threshold=1,           # 1 confirm: candidate → provisional
    user_assertion_auto_promote=True,   # "I am vegetarian" → trusted immediately
    confidence_floor=0.2,              # Below this → deprecated
    staleness_days=180,                # Unconfirmed for 6 months → review
)

engine = PromotionEngine(config)
print(f"Promotion engine configured:")
print(f"  Provisional after: {config.provisional_threshold} confirmation(s)")
print(f"  Trusted after:     {config.confirmation_threshold} confirmation(s)")
print(f"  Auto-promote user assertions: {config.user_assertion_auto_promote}")
print(f"  Deprecate below confidence:   {config.confidence_floor}")
print(f"  Staleness review after:       {config.staleness_days} days")

## Demo: The Vegetarian Scenario

Watch how a memory progresses through states as the user confirms it over
multiple interactions:

1. **Turn 1**: Agent infers "user might be vegetarian" from a meal choice → `CANDIDATE`
2. **Turn 3**: User orders vegetarian again → confirmed once → `PROVISIONAL`
3. **Turn 5**: User explicitly says "I'm vegetarian" → confirmed twice → `TRUSTED`

Critically: the agent does NOT recommend steakhouses while the memory is still `CANDIDATE`.

In [ ]:
# Create a memory from LLM inference (lowest authority)
vegetarian_memory = MemoryItem(
    user_id="E001",
    content="User prefers vegetarian food",
    category="preference",
    state=MemoryState.CANDIDATE,
    confidence=0.5,
    source_type="llm_inference",
)

print(f"=== Initial State ===")
print(f"Memory: {vegetarian_memory.content}")
print(f"State:  {vegetarian_memory.state.value}")
print(f"Confirmations: {vegetarian_memory.confirmation_count}")
print(f"Should agent use this? {'YES' if vegetarian_memory.state == MemoryState.TRUSTED else 'NO'}")

In [ ]:
# Turn 3: User orders vegetarian meal → first confirmation signal
print("=== Signal: User ordered a vegetarian meal ===")
vegetarian_memory.confirm()
changed = engine.apply(vegetarian_memory)
print(f"State:  {vegetarian_memory.state.value} (changed={changed})")
print(f"Confirmations: {vegetarian_memory.confirmation_count}")
print(f"Should agent use this? {'YES (with hedging)' if vegetarian_memory.state == MemoryState.PROVISIONAL else 'NO'}")
print(f"State history: {json.dumps(vegetarian_memory.state_history, indent=2)}")

In [ ]:
# Turn 5: User explicitly confirms → second confirmation
print("=== Signal: User says 'I'm vegetarian' ===")
vegetarian_memory.confirm()
changed = engine.apply(vegetarian_memory)
print(f"State:  {vegetarian_memory.state.value} (changed={changed})")
print(f"Confirmations: {vegetarian_memory.confirmation_count}")
print(f"Should agent use this? {'YES — full confidence' if vegetarian_memory.state == MemoryState.TRUSTED else 'NO'}")
print(f"\nFull state history:")
for entry in vegetarian_memory.state_history:
    print(f"  {entry['from']} → {entry['to']} ({entry['reason']})")

## Agent Behaviour Gating

The key insight: the agent's behaviour changes based on memory state.
Only `TRUSTED` memories are used as facts. `PROVISIONAL` memories use hedging
language. `CANDIDATE` memories don't appear in recommendations at all.

In [ ]:
from agent_framework import Agent, AgentSession

def build_memory_context(memories: list[MemoryItem]) -> str:
    """Build context string only from trusted/provisional memories."""
    lines = []
    for m in memories:
        if m.state == MemoryState.TRUSTED:
            lines.append(f"[KNOWN FACT] {m.content}")
        elif m.state == MemoryState.PROVISIONAL:
            lines.append(f"[LIKELY — confirm before assuming] {m.content}")
        # CANDIDATE memories are intentionally excluded!
    return "\n".join(lines) if lines else "No personalisation data available."


# Demonstrate with different states
test_memories = [
    MemoryItem(content="Prefers Marriott hotels", state=MemoryState.TRUSTED, user_id="E001"),
    MemoryItem(content="Prefers vegetarian food", state=MemoryState.PROVISIONAL, user_id="E001"),
    MemoryItem(content="Might prefer morning flights", state=MemoryState.CANDIDATE, user_id="E001"),
]

context = build_memory_context(test_memories)
print("Memory context injected into agent:")
print(context)
print("\n(Notice: CANDIDATE memory about morning flights is NOT included)")

In [ ]:
# Run the agent with memory-gated context
memory_gated_agent = Agent(
    client=client,
    name="MemoryGatedAssistant",
    instructions=SYSTEM_PROMPT + f"\n\nUser personalisation:\n{context}",
    tools=[],
)

async def demo_gated_response():
    session = AgentSession()
    query = "Can you recommend a restaurant near my hotel for dinner tonight?"
    print(f"Query: {query}\n")
    result = await memory_gated_agent.run(query, session=session)
    print(f"Agent: {result.text}")
    print("\n--- Notice how the agent:")
    print("  • Uses Marriott fact with confidence (TRUSTED)")
    print("  • Mentions vegetarian with hedging/confirmation (PROVISIONAL)")
    print("  • Does NOT mention morning flights (CANDIDATE — filtered out)")

asyncio.run(demo_gated_response())

## User Assertion: Auto-Promotion

When the user explicitly states something ("I am vegetarian"), it's the highest
authority source and skips directly to `TRUSTED`.

In [ ]:
# Direct user assertion — highest authority
user_stated = MemoryItem(
    user_id="E001",
    content="User is vegetarian",
    category="fact",
    state=MemoryState.CANDIDATE,
    confidence=0.9,
    source_type="user_assertion",  # Key difference!
)

print(f"Before: state={user_stated.state.value}, source={user_stated.source_type}")
engine.apply(user_stated)
print(f"After:  state={user_stated.state.value}")
print(f"→ User assertions skip the confirmation queue entirely")

## Poisoning Defense Demo

Memory poisoning occurs when adversarial information gets injected (via tool output,
prompt injection, or cross-session contamination). Staged promotion defends against this
because injected memories:

1. Enter as `CANDIDATE` (never immediately trusted)
2. Receive no confirmation signals (user never validates them)
3. Never influence agent behaviour
4. Eventually decay or get evicted

In [ ]:
# Simulate memory poisoning: injected via a tool output
poisoned_memory = MemoryItem(
    user_id="E001",
    content="User loves extremely spicy food and always wants the hottest option",
    category="preference",
    state=MemoryState.CANDIDATE,
    confidence=0.4,  # Lower confidence from tool output
    source_type="tool_output",  # Not user assertion!
)

print("=== Poisoned Memory Injected ===")
print(f"Content: {poisoned_memory.content}")
print(f"State:   {poisoned_memory.state.value}")
print(f"Source:  {poisoned_memory.source_type}")
print()

# Evaluate — should NOT promote (no confirmations, not user assertion)
changed = engine.apply(poisoned_memory)
print(f"After promotion evaluation: state={poisoned_memory.state.value} (changed={changed})")
print(f"Confirmations: {poisoned_memory.confirmation_count}")
print()

# Check if it would appear in agent context
context = build_memory_context([poisoned_memory])
print(f"Agent context: '{context}'")
print("\n✅ Poisoned memory has NO influence on agent behaviour!")
print("   It sits in CANDIDATE state with no path to TRUSTED.")

In [ ]:
# What if the poisoned memory's confidence drops further?
print("=== Confidence Decay Over Time ===")
poisoned_memory.confidence = 0.15  # Dropped below floor
changed = engine.apply(poisoned_memory)
print(f"Confidence dropped to {poisoned_memory.confidence}")
print(f"State: {poisoned_memory.state.value} (changed={changed})")
print(f"\n→ Eventually deprecated and removed from the store entirely")
print(f"\nState history:")
for entry in poisoned_memory.state_history:
    print(f"  {entry['from']} → {entry['to']} ({entry['reason']})")

## Staleness Demotion

Even `TRUSTED` memories can be demoted if they haven't been confirmed in a long time.
People's preferences change — a 2-year-old preference might no longer be valid.

In [ ]:
# Create a trusted memory that hasn't been confirmed in 200 days
stale_memory = MemoryItem(
    user_id="E001",
    content="User prefers Budget car rental",
    category="preference",
    state=MemoryState.TRUSTED,
    confidence=0.8,
    confirmation_count=3,
    source_type="llm_inference",
    last_confirmed=datetime.now(timezone.utc) - timedelta(days=200),
)

print(f"Memory: {stale_memory.content}")
print(f"State:  {stale_memory.state.value}")
print(f"Last confirmed: {stale_memory.last_confirmed.date()} ({200} days ago)")
print(f"Staleness threshold: {config.staleness_days} days\n")

changed = engine.apply(stale_memory)
print(f"After staleness check: state={stale_memory.state.value} (changed={changed})")
print(f"→ Demoted to PROVISIONAL — agent will ask for confirmation next time")

## Summary: Trust Levels and Agent Behaviour

| State | Used in Decisions? | Language Style | Promotion Path |
|-------|-------------------|----------------|----------------|
| `CANDIDATE` | ❌ Never | N/A (invisible to agent) | Confirm → PROVISIONAL |
| `PROVISIONAL` | ⚠️ With hedging | "I think you might prefer..." | Confirm → TRUSTED |
| `TRUSTED` | ✅ As fact | "Based on your preference for..." | Staleness → PROVISIONAL |
| `DEPRECATED` | ❌ Never | N/A (historical only) | — |

## Key Takeaways

1. **Never trust immediately** — all inferred memories start as CANDIDATE
2. **User assertions are privileged** — explicit statements skip the queue
3. **Behaviour gating prevents poisoning** — unconfirmed memories can't influence decisions
4. **Staleness keeps memories fresh** — old trusted memories get re-evaluated
5. **State history provides audit trail** — full provenance of trust decisions

## Next: Belief Revision (Notebook 04)

What happens when a trusted memory becomes *wrong*? People move cities, change
preferences, update facts. The next notebook implements structured belief revision
with temporal validity tracking.